# Домашнее задание 4.3 — Бассейны Ньютона при слиянии корней (🟡)

Семейство полиномов

$$
f_\varepsilon(z)=(z+1)(z-1)(z-\varepsilon),\qquad z\in\mathbb C.
$$

При $\varepsilon\to1$ корень $z=\varepsilon$ приближается к $z=1$. При $\varepsilon=1$ они образуют двойной корень. Строю бассейны для четырёх значений параметра $\varepsilon=0.5,\ 0.9,\ 0.99,\ 1$ по итерации

$$
z_{n+1}=z_n-\frac{f_\varepsilon(z_n)}{f_\varepsilon'(z_n)}.
$$

Как на зелёном уровне, беру область $\mathrm{Re}\,z,\mathrm{Im}\,z\in[-2,2]$, сетку $600\times600$ и базовый лимит 100 итераций. Сравниваю бассейны и отдельно исследую, как ошибка и невязка убывают около простого и двойного корней.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.animation import PillowWriter
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

plt.rcParams["figure.dpi"] = 110

## 1. Классификация исходов и допуски

| код | исход | условие |
|---|---|---|
| $0,1,2$ | сошлась к $-1$, $1$ или $\varepsilon$ | $\lvert z_n-r_k\rvert<\delta_z$ и $\lvert f_\varepsilon(z_n)\rvert<\delta_f$ |
| $-2$ | численный сбой | $\lvert f_\varepsilon'(z_n)\rvert<\delta_{f'}$ или нечисловое либо бесконечное значение при вычислении |
| $-1$ | не определено | за $N_{\max}$ шагов исход не установлен |

В основном расчёте $\delta_z=10^{-8}$, $\delta_f=10^{-10}$ и $\delta_{f'}=10^{-14}$. Наименьшее ненулевое расстояние между корнями равно $0.01$, поэтому окрестности классификации разделены с большим запасом. При $\varepsilon=1$ двойной корень получает код 1. В вычислении $f_\varepsilon$ множитель $(z-1)$ учитывается дважды.

На каждом слое сначала проверяю сходимость и только потом проверяю производную перед делением. Поэтому начальная точка точно в двойном корне распознаётся сразу. Из зелёного уровня сохраняю расчёт только по активным точкам.

In [ ]:
UNDETERMINED, FAILURE = -1, -2
TOL_Z, TOL_F, TOL_D = 1e-8, 1e-10, 1e-14


def roots_of(eps):
    return np.array([-1.0, 1.0, eps], dtype=complex)


def distinct_roots(eps):
    roots = roots_of(eps)
    return roots[:2] if eps == 1 else roots


def f_and_df(z, roots):
    d = z[..., None] - roots
    f = np.prod(d, axis=-1)
    df = (d[..., 1] * d[..., 2] + d[..., 0] * d[..., 2] + d[..., 0] * d[..., 1])
    return f, df


def newton_classify(Z0, eps, n_max=100, tol_z=TOL_Z, tol_f=TOL_F, tol_d=TOL_D):
    roots = roots_of(eps)
    targets = distinct_roots(eps)
    shape = np.shape(Z0)
    z = np.array(Z0, dtype=complex).ravel().copy()
    code = np.full(z.size, UNDETERMINED, dtype=int)
    n_iter = np.full(z.size, n_max, dtype=int)
    idx = np.arange(z.size)

    for n in range(n_max + 1):
        zi = z[idx]
        f, df = f_and_df(zi, roots)

        dist = np.abs(zi[:, None] - targets[None, :])
        k = np.argmin(dist, axis=1)
        conv = (dist[np.arange(len(idx)), k] < tol_z) & (np.abs(f) < tol_f)
        code[idx[conv]] = k[conv]
        n_iter[idx[conv]] = n

        finite = np.isfinite(zi) & np.isfinite(f) & np.isfinite(df)
        bad = ~conv & ((np.abs(df) < tol_d) | ~finite)
        code[idx[bad]] = FAILURE
        n_iter[idx[bad]] = n

        keep = ~(conv | bad)
        if n == n_max or not keep.any():
            break
        idx, zi, f, df = idx[keep], zi[keep], f[keep], df[keep]
        z[idx] = zi - f / df

    return code.reshape(shape), n_iter.reshape(shape), z.reshape(shape)

## 2. Карты бассейнов и анимация

Цвета соответствуют одним и тем же исходам на всех картах. Слева в каждой паре показана вся область начальных приближений, справа — её фрагмент около $z_0=1$ в одном и том же масштабе. Квадрат отмечает движущийся корень $\varepsilon$, круги — неподвижные корни. При слиянии остаётся один круг в точке $1$.

Чёрным обозначены точки, для которых не хватило итераций, пурпурным — численные сбои.

In [ ]:
N, EXT = 600, 2.0
EPS_VALUES = [0.5, 0.9, 0.99, 1.0]
x = np.linspace(-EXT, EXT, N)
Z0 = x[None, :] + 1j * x[:, None]
half_pixel = (x[1] - x[0]) / 2
extent = (-EXT - half_pixel, EXT + half_pixel) * 2

runs = {eps: newton_classify(Z0, eps)[:2] for eps in EPS_VALUES}
ROOT_COLORS = ["#4c72b0", "#dd8452", "#55a868"]
CAT_COLORS = {UNDETERMINED: "#000000", FAILURE: "#ff00ff"}
cmap = ListedColormap(ROOT_COLORS + [CAT_COLORS[UNDETERMINED], CAT_COLORS[FAILURE]])
ZOOM = (0.87, 1.05, -0.09, 0.09)


def show_basins(ax, outcomes, eps, zoom=False):
    img = np.where(outcomes >= 0, outcomes, 2 - outcomes)
    ax.imshow(img, extent=extent, origin="lower", cmap=cmap,
              vmin=-0.5, vmax=4.5, interpolation="nearest")
    for k, r in enumerate(distinct_roots(eps)):
        ax.plot(r.real, r.imag, "s" if k == 2 else "o", ms=5 if zoom else 6,
                mfc="white", mec="black", mew=0.9)
    if zoom:
        ax.axis(ZOOM)
        ax.set_xticks([0.90, 0.95, 1.00])
        ax.set_yticks([-0.05, 0, 0.05])
        ax.set_title("Окрестность 1", fontsize=10)
    else:
        ax.axis((-EXT, EXT, -EXT, EXT))
        ax.set_xticks(np.arange(-2, 3))
        ax.set_yticks(np.arange(-2, 3))
        ax.set_xlabel(r"$\mathrm{Re}\,z_0$")
        ax.set_ylabel(r"$\mathrm{Im}\,z_0$")
        title = rf"$\varepsilon = {eps:g}$"
        if eps == 1:
            title += "  (двойной корень)"
        ax.set_title(title, fontsize=11)
        left, right, bottom, top = ZOOM
        ax.add_patch(Rectangle((left, bottom), right - left, top - bottom,
                               fill=False, ec="black", lw=0.75))
    ax.set_aspect("equal")
    ax.tick_params(labelsize=8)


legend = [
    Patch(color=ROOT_COLORS[0], label=r"$-1$"),
    Patch(color=ROOT_COLORS[1], label=r"$1$ (двойной при $\varepsilon=1$)"),
    Patch(color=ROOT_COLORS[2], label=r"$\varepsilon$ (при $\varepsilon<1$)"),
    Patch(color=CAT_COLORS[UNDETERMINED], label="не определено"),
    Patch(color=CAT_COLORS[FAILURE], label="сбой"),
    Line2D([], [], ls="", marker="o", mfc="white", mec="black", label="положение корня"),
]

fig, axes = plt.subplots(2, 4, figsize=(11.4, 8.4), width_ratios=[1, 0.47, 1, 0.47])
fig.suptitle(r"Бассейны Ньютона, сетка $600\times600$, $N_{\max}=100$")
for panel, ((ax, zoom_ax), eps) in enumerate(zip(axes.reshape(-1, 2), EPS_VALUES)):
    outcomes, _ = runs[eps]
    show_basins(ax, outcomes, eps)
    show_basins(zoom_ax, outcomes, eps, zoom=True)
    if panel < 2:
        ax.set_xlabel("")
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=9)
fig.tight_layout(rect=(0, 0.085, 1, 0.96), h_pad=2)
plt.show()

**Анимация.** Четыре кадра показывают те же расчёты по мере увеличения $\varepsilon$. Область и цвета зафиксированы, чтобы можно было сравнивать границы бассейнов между кадрами.

In [ ]:
fig, (ax, zoom_ax) = plt.subplots(1, 2, figsize=(7.6, 4.7), width_ratios=[1, 0.48])
fig.suptitle("Сближение корней и изменение бассейнов")
fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=8)
fig.subplots_adjust(left=0.10, right=0.96, bottom=0.23, top=0.88, wspace=0.38)

with TemporaryDirectory() as temp_dir:
    gif_path = Path(temp_dir) / "newton_basins.gif"
    writer = PillowWriter(fps=1)
    with writer.saving(fig, gif_path, dpi=105):
        for eps in EPS_VALUES:
            ax.clear()
            zoom_ax.clear()
            outcomes, _ = runs[eps]
            show_basins(ax, outcomes, eps)
            show_basins(zoom_ax, outcomes, eps, zoom=True)
            writer.grab_frame()
    animation = Image(data=gif_path.read_bytes(), format="gif", width=760)
plt.close(fig)
display(animation)

## 3. Влияние лимита итераций и допусков

Сначала привожу число точек каждого исхода в основном расчёте. Затем меняю по одному параметру относительно него. Уменьшаю лимит до 20 шагов и увеличиваю до 200. Отдельно ослабляю допуск по расстоянию или по невязке до $10^{-4}$. Даже ослабленный допуск по расстоянию в 100 раз меньше минимального ненулевого зазора между корнями. Допуск на производную остаётся фиксированным.

Столбец «Изменили код» показывает число точек, чей исход отличается от основного расчёта. Медиану и максимум считаю только по точкам, удовлетворившим обоим критериям сходимости.

In [ ]:
print(f"{'ε':>5} {'−1':>8} {'1':>8} {'ε':>8} {'не опр.':>8} {'сбой':>7}")
for eps, (outcomes, _) in runs.items():
    moving = str(np.count_nonzero(outcomes == 2)) if eps < 1 else "—"
    counts = [np.count_nonzero(outcomes == k) for k in (0, 1, UNDETERMINED, FAILURE)]
    print(f"{eps:5g} {counts[0]:8d} {counts[1]:8d} {moving:>8}"
          f" {counts[2]:8d} {counts[3]:7d}")

experiments = [
    ("основной", {}),
    ("N_max = 20", {"n_max": 20}),
    ("N_max = 200", {"n_max": 200}),
    ("δ_z = 1e−4", {"tol_z": 1e-4}),
    ("δ_f = 1e−4", {"tol_f": 1e-4}),
]

print()
print(f"{'ε':>5} {'Режим':>13} {'Не опр.':>8} {'Сбой':>6}"
      f" {'Изменили код':>13} {'Медиана':>8} {'Макс.':>6}")
for eps, (base_code, base_iters) in runs.items():
    for label, options in experiments:
        outcomes, nit = newton_classify(Z0, eps, **options)[:2] if options else (base_code, base_iters)
        converged = nit[outcomes >= 0]
        print(f"{eps:5g} {label:>13} {np.count_nonzero(outcomes == UNDETERMINED):8d}"
              f" {np.count_nonzero(outcomes == FAILURE):6d} {np.count_nonzero(outcomes != base_code):13d}"
              f" {np.median(converged):8.0f} {converged.max():6d}")

Медиана числа шагов при базовых допусках растёт с 7 при $\varepsilon=0.5$ до 27 при $\varepsilon=1$. Для двойного корня особенно заметно влияние допуска по расстоянию. Если ослабить только его, медиана снижается с 27 до 18. Если ослабить только допуск по невязке, медиана остаётся равной 27.

При $N_{\max}=20$ и $\varepsilon=1$ исход остаётся неопределённым для 250 044 точек. При лимите 100 все точки этой сетки классифицируются, а при 200 исходы совпадают с результатом при 100.

## 4. Ошибка и невязка на траекториях

Из одной начальной точки $z_0=1.05+0.04i$ считаю две траектории к известному корню $r=1$. При $\varepsilon=0.99$ корень простой, при $\varepsilon=1$ — двойной. На каждой итерации вычисляю

$$
e_n=|z_n-1|,\qquad \rho_n=|f_\varepsilon(z_n)|.
$$

Для графиков продолжаю итерации за пределами базового критерия остановки, используя $\delta_z=10^{-13}$, $\delta_f=10^{-16}$ и лимит 40 шагов. Вертикальный пунктир отмечает шаг, на котором основной расчёт уже остановился бы. Фоном выделены данные для оценки порядка, выбранные до порога диагностической остановки и до влияния округления.

In [ ]:
def newton_path(z0, eps, n_max=40, tol_z=1e-13, tol_f=1e-16):
    roots = roots_of(eps)
    z = complex(z0)
    path = [z]
    for _ in range(n_max):
        f, df = f_and_df(np.asarray(z), roots)
        if np.min(np.abs(z - roots)) < tol_z and abs(f) < tol_f:
            break
        if not np.isfinite([z, f, df]).all() or abs(df) < TOL_D:
            break
        z -= f / df
        path.append(z)
    return np.array(path)


z_start = 1.05 + 0.04j
traces = {}
windows = {0.99: (5, 7), 1.0: (9, 19)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharey=True)
for ax, eps, title in zip(axes, windows, ["Простой корень", "Двойной корень"]):
    path = newton_path(z_start, eps)
    e = np.abs(path - 1)
    rho = np.abs(f_and_df(path, roots_of(eps))[0])
    traces[eps] = e, rho
    stop = np.flatnonzero((e < TOL_Z) & (rho < TOL_F))[0]
    ax.axvspan(*windows[eps], color="0.93", label="окно оценки порядка")
    ax.semilogy(e, "o-", ms=3, label=r"$e_n=|z_n-1|$")
    ax.semilogy(rho, "s-", ms=3, label=r"$\rho_n=|f_\varepsilon(z_n)|$")
    ax.axhline(TOL_Z, color="C0", ls=":", lw=1)
    ax.axhline(TOL_F, color="C1", ls=":", lw=1)
    ax.axvline(stop, color="0.35", ls="--", lw=1, label=f"базовая остановка, n = {stop}")
    ax.set_title(rf"{title}, $\varepsilon={eps:g}$")
    ax.set_xlabel("номер итерации n")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)

axes[0].set_ylabel("ошибка и невязка (лог. шкала)")
fig.tight_layout()
plt.show()

## 5. Численная оценка порядка сходимости

Если $e_{n+1}\approx C e_n^p$, то по тройке последовательных ошибок можно оценить порядок

$$
p_n=\frac{\log(e_{n+1}/e_n)}{\log(e_n/e_{n-1})}.
$$

Для простого корня использую шаги 5–7. Здесь ошибка уже существенно меньше зазора $0.01$, а $e_7\approx4.6\cdot10^{-9}$ ещё далека от машинной точности. Ошибку $e_8\approx2.1\cdot10^{-15}$ в оценку не включаю. Для двойного корня беру шаги 9–19, где начальный переход уже прошёл и ошибки ещё далеки как от порога остановки, так и от округления.

Дополнительно сравниваю $e_{n+1}/e_n^2$ у простого корня и $e_{n+1}/e_n$ у двойного с ожидаемыми предельными значениями. Из разложения шага Ньютона вблизи простого корня $r$ получаю $C=|f_\varepsilon''(r)/(2f_\varepsilon'(r))|$. Для двойного корня $f_1(z)=(z+1)(z-1)^2$, поэтому шаг Ньютона в первом приближении уменьшает отклонение вдвое.

In [ ]:
for eps, (e, _) in traces.items():
    lo, hi = windows[eps]
    n = np.arange(lo + 1, hi)
    p = np.log(e[n + 1] / e[n]) / np.log(e[n] / e[n - 1])
    degree = 2 if eps < 1 else 1
    q = e[lo + 1:hi + 1] / e[lo:hi]**degree
    expected = (3 - eps) / (2 * (1 - eps)) if eps < 1 else 0.5
    print(f"ε = {eps:g}, окно n = {lo}…{hi}")
    print(f"  p: медиана {np.median(p):.4f}, диапазон [{p.min():.4f}, {p.max():.4f}]")
    print(f"  e_(n+1) / e_n^{degree}: диапазон [{q.min():.4f}, {q.max():.4f}],"
          f" теоретический предел {expected:g}")

print()
print(f"{'ε':>5} {'Критерий':>23} {'n':>5} {'Ошибка':>12} {'Невязка':>12}")
for eps, (e, rho) in traces.items():
    for label, accepted in [
        ("только невязка", rho < TOL_F),
        ("оба условия", (rho < TOL_F) & (e < TOL_Z)),
    ]:
        n = np.flatnonzero(accepted)[0]
        print(f"{eps:5g} {label:>23} {n:5d} {e[n]:12.3e} {rho[n]:12.3e}")

У простого корня оценка $p_6\approx2.01$ согласуется с квадратичной сходимостью. Отношение $e_7/e_6^2\approx100.4$ близко к теоретическому пределу $100.5$. В начале траектории расстояние до корня ещё сопоставимо с зазором между корнями, и квадратичный режим устанавливается лишь после приближения к $1$. У двойного корня $p\approx1$ и $e_{n+1}/e_n\approx0.5$, что соответствует линейной сходимости.

## 6. Ответ на вопрос

**Насколько малая невязка гарантирует близость к корню, когда два корня почти слились?**

При $\varepsilon<1$ и $|z-1|\ll1-\varepsilon$ из разложения по множителям получаю

$$
|f_\varepsilon(z)|\approx 2(1-\varepsilon)|z-1|.
$$

Коэффициент при ошибке стремится к нулю. Поэтому одна и та же малая невязка около корня $1$ допускает всё большую ошибку по мере сближения корней. Когда ошибка сравнима с зазором, линейная оценка уже неприменима. Например, в середине между корнями при $\varepsilon=0.99$ расстояние до каждого из них равно $0.005$, а невязка около $5\cdot10^{-5}$.

При слиянии $|f_1(z)|\approx 2|z-1|^2$. В эксперименте с двойным корнем условие $\rho_n<10^{-10}$ впервые выполнено на шаге 14 при ошибке около $4.0\cdot10^{-6}$. Чтобы достичь ошибки меньше $10^{-8}$, понадобилось 23 шага. Поэтому на картах проверяю и невязку, и расстояние до конкретного корня.